In [24]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/PB_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)
# ===============================
# Define Features
# ===============================

features = ['NDVI', 'NDMI', 'NBR', 'BSI']

# Keep only one row per location-year
df = df.groupby(
    ['latitude', 'longitude', 'year']
)[features].mean().reset_index()

print("After grouping shape:", df.shape)
print(df.head())
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
After grouping shape: (51, 7)
    latitude  longitude  year      NDVI      NDMI       NBR       BSI
0  29.740973  75.383475  2022  0.404197  0.169532  0.317854 -0.082998
1  29.740973  75.383475  2023  0.408649  0.175507  0.316400 -0.108300
2  29.740973  75.383475  2024  0.360656  0.125502  0.241642 -0.057347
3  29.901772  75.572121  2022  0.326464  0.144466  0.263537 -0.055339
4  29.901772  75.572121  2023  0.343806  0.183846  0.303088 -0.091330
    latitude  longitude  year      NDVI      NDMI       NBR       BSI
0  29.740973  75.383475  2022  0.404197  0.169532  0.317854 -0.082998
1  29.740973  75.383475  2023  0.408649  0.175507  0.316400 -0.108300
2  29.740973  75.383475  2024  0.360656  0.125502  0.241642 -0.057347
3  29.901772  75.572121  2022  0.326464  0.144466  0.263537 -0.055339
4  29.901772  75.572121  2023  0.343806  0.183846  0.303088 -0.091330
[

In [25]:
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

Raw input shape: (0,)


In [26]:
# ===============================
# Create Proper LSTM Structure
# ===============================

features = ['NDVI', 'NDMI', 'NBR', 'BSI']

# Ensure correct year order
df = df.sort_values(['latitude','longitude','year'])

years = sorted(df['year'].unique())
print("Years:", years)

locations = df[['latitude','longitude']].drop_duplicates().values

X_list = []

for lat, lon in locations:
    temp = df[(df['latitude']==lat) & (df['longitude']==lon)]

    # Ensure full time sequence exists
    if len(temp) == len(years):
        X_list.append(temp[features].values)

X_raw = np.array(X_list)

print("X_raw shape:", X_raw.shape)

Years: [np.int64(2022), np.int64(2023), np.int64(2024)]
X_raw shape: (17, 3, 4)


In [27]:

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/PB_Deforestation_Predictions.csv',
    index=False
)

print("Saved:PB_Deforestation_Predictions.csv")



Label distribution:
No deforestation: 15
Deforestation: 2
Train shape: (13, 3, 4)
Test shape : (4, 3, 4)
Scaled train shape: (13, 3, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.6923 - loss: 1.8539 - precision: 0.2500 - recall: 0.5000 - val_accuracy: 1.0000 - val_loss: 0.6711 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.8462 - loss: 1.8483 - precision: 0.5000 - recall: 0.5000 - val_accuracy: 1.0000 - val_loss: 0.6785 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - accuracy: 0.6154 - loss: 1.8373 - precision: 0.2857 - recall: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.6855 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.6154 - loss: 1.8432 - precision: 0.2857 - recall: 1.0000 - val_accuracy: 0.5000 - val_loss: 0.6924 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.6154 - loss: 1.8212 - precision: 0.2857 - recall: 1.0000 - val_accuracy: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [28]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/PB_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/PB_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/PB_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(0.0018665876470588196), 'std': 0.04001245915682882, 'q1': np.float64(-0.017722600000000033), 'q3': np.float64(0.02726841999999996), 'lower_2std': np.float64(-0.07815833066659882), 'upper_2std': np.float64(0.08189150596071647)}
NBR : {'mean': np.float64(0.005074432352941174), 'std': 0.06136763845003932, 'q1': np.float64(-0.027768700000000035), 'q3': np.float64(0.033641019999999966), 'lower_2std': np.float64(-0.11766084454713746), 'upper_2std': np.float64(0.12780970925301982)}
BSI : {'mean': np.float64(-0.008762299019333612), 'std': 0.03547565713083679, 'q1': np.float64(-0.0463038932705214), 'q3': np.float64(0.021754423767524192), 'lower_2std': np.float64(-0.07971361328100718), 'upper_2std': np.float64(0.06218901524233996)}
NDMI: {'mean': np.float64(0.00024940258823529367), 'std': 0.04497906702814214, 'q1': np.float64(-0.03636930999999999), 'q3': np.float64(0.03885448999999999), 'lower_2std': np.float64(-0.08970873146804899), 'upper_2std': np.float64(0.090207536

In [30]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/PB_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Agriculture    2
Urban/Other    2
Name: count, dtype: int64


In [31]:

m.save('/content/drive/MyDrive/pb_adaptive.html')


In [32]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS (PB)
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/PB_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA (PB)
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/PB_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD PUNJAB DISTRICTS
# ==============================

pb = gpd.read_file('/content/drive/MyDrive/Punjab.geojson')

# CRS SAFETY
if pb.crs is None:
    pb.set_crs("EPSG:4326", inplace=True)

pb = pb.to_crs("EPSG:4326")

# Add state label (optional)
pb["state"] = "Punjab"

print(pb.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 4
    latitude  longitude  deforestation
0  30.377879  75.380780              1
1  30.198216  74.402514              1
2  30.412015  74.443837              1
3  30.300624  74.480668              1
Total deforestation samples: 4
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  30.377879  75.380780              1    -0.034044   -0.025737    0.015184   
1  30.198216  74.402514              1     0.064679    0.141205   -0.052158   
2  30.412015  74.443837              1     0.027268    0.019323   -0.016312   
3  30.300624  74.480668              1    -0.017723   -0.027769    0.023183   

   NDMI_change        cause  
0    -0.015808  Agriculture  
1     0.056086  Urban/Other  
2     0.007439  Urban/Other  
3    -0.036369  Agriculture  
  REMARKS_2 Country State_Name State_Code        Dist_Name Dist_Code  \
0      None   India     Punjab         03         Amritsar       049   
1      None   India     Punjab         03         Bathinda       04

In [33]:
# ==============================
# SPATIAL JOIN (Points → Punjab Districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    pb,                # Punjab GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — PB
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/PB_District_Wise_Deforestation.csv',
    index=False
)

print("Saved PB district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY — PB
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/PB_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved PB cause summary")

    latitude  longitude  deforestation        cause  \
0  30.377879  75.380780              1  Agriculture   
1  30.198216  74.402514              1  Urban/Other   
2  30.412015  74.443837              1  Urban/Other   
3  30.300624  74.480668              1  Agriculture   

                    geometry  index_right REMARKS_2 Country State_Name  \
0  POINT (75.38078 30.37788)           18      None   India     Punjab   
1  POINT (74.40251 30.19822)           11      None   India     Punjab   
2  POINT (74.44384 30.41201)           11      None   India     Punjab   
3  POINT (74.48067 30.30062)           11      None   India     Punjab   

  State_Code Dist_Name Dist_Code   state  
0         03   Barnala       054  Punjab  
1         03   Muktsar       044  Punjab  
2         03   Muktsar       044  Punjab  
3         03   Muktsar       044  Punjab  
Total joined points: 4
Points without district: 0
Dist_Name
Muktsar    3
Barnala    1
Name: count, dtype: int64
  Dist_Name  deforestation

In [34]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Punjab District Boundaries
# =====================================================
pb = gpd.read_file('/content/drive/MyDrive/Punjab.geojson')

if pb.crs is None:
    pb.set_crs(epsg=4326, inplace=True)

pb = pb.to_crs(epsg=4326)
pb["state"] = "Punjab"
gdf_districts = pb.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/PB_Deforestation_Predictions.csv")

if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats
# =====================================================
gdf_districts = gdf_districts.merge(district_total, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="Dist_Name", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (Center Punjab)
# =====================================================
m = folium.Map(location=[31.0, 75.0], zoom_start=7)

# =====================================================
# STEP 10: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Punjab)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Punjab Afforestation Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Punjab Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Promote plantation and sustainable land management.
"""

folium.Marker(
    location=[31.0, 75.0],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/pb_forest.html")

print("✅ Punjab map saved successfully with dynamic afforestation alert!")

/tmp/ipython-input-403/1914331065.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Punjab map saved successfully with dynamic afforestation alert!
